# Phase 1 — Data Acquisition & Preparation
**Steps 1.1 – 1.5** | Inspect, clean, recode, and slice the UNGA voting data.

Data sources:
- `kaggle_files/votes.csv` — country × resolution × vote (1=yes, 2=abstain, 3=no, 9=absent/non-member)
- `kaggle_files/resolutions.csv` — resolution metadata + 6 issue flags
- `kaggle_files/states.csv` — country-level ideal-point estimates & affinity scores
- `dataverse_files/AgreementScores.csv` — pre-computed pairwise agreement (ccode pairs, year)

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

# ── Portable root path ──────────────────────────────────────────────────────
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
print('Project root:', ROOT)

RAW     = os.path.join(ROOT, 'kaggle_files')
DV      = os.path.join(ROOT, 'dataverse_files')
PROC    = os.path.join(ROOT, 'data', 'processed')
EXT     = os.path.join(ROOT, 'data', 'external')
PLOTS   = os.path.join(ROOT, 'results', 'plots')
TABLES  = os.path.join(ROOT, 'results', 'tables')

for d in [PROC, EXT, PLOTS, TABLES]:
    os.makedirs(d, exist_ok=True)

## Step 1.1 – Load raw files

In [ ]:
# ── Votes ────────────────────────────────────────────────────────────────────
votes_raw = pd.read_csv(os.path.join(RAW, 'votes.csv'))
print('votes_raw shape:', votes_raw.shape)
print(votes_raw.dtypes)
votes_raw.head()

In [ ]:
# ── Resolutions (metadata + issue flags) ────────────────────────────────────
res_raw = pd.read_csv(os.path.join(RAW, 'resolutions.csv'))
print('resolutions shape:', res_raw.shape)
print(res_raw.columns.tolist())
res_raw.head()

In [ ]:
# ── States (country metadata, ideal-points) ─────────────────────────────────
states_raw = pd.read_csv(os.path.join(RAW, 'states.csv'))
print('states shape:', states_raw.shape)
states_raw.head()

## Step 1.2 – Understand Schema

In [ ]:
# Vote distribution
print('Vote value counts (1=yes, 2=abstain, 3=no, 8=absent, 9=non-member):')
print(votes_raw['vote'].value_counts())

# Year range (assembly_session → year mapping from states)
session_year = states_raw[['assembly_session','year']].drop_duplicates()
print(f'\nSessions: {session_year["assembly_session"].min()} – {session_year["assembly_session"].max()}')
print(f'Years:    {session_year["year"].min()} – {session_year["year"].max()}')

# Issue flag columns
ISSUES = {
    'co': 'colonization',
    'hr': 'human_rights',
    'me': 'israel_palestine',
    'di': 'disarmament',
    'nu': 'nuclear_weapons',
    'ec': 'economic_development'
}
print('\nResolution issue flag sums:')
for short, col in ISSUES.items():
    if col in res_raw.columns:
        print(f'  {short} ({col}): {int(res_raw[col].sum())}')

## Step 1.3 – Clean & Recode Votes

In [ ]:
# Merge votes with session→year mapping
df = votes_raw.merge(session_year, on='assembly_session', how='left')

# Drop absent (8) and non-member (9) — Kaggle uses 8 & 9 interchangeably here as 8/9
# Kaggle votes: 1=yes, 2=abstain, 3=no, 8=abdrain, 9=non-member
df = df[df['vote'].isin([1, 2, 3])].copy()
print('After dropping absent/non-member:', df.shape)

# Recode: 1→1 (yes), 2→0 (abstain), 3→-1 (no)
vote_recode = {1: 1, 2: 0, 3: -1}
df['v'] = df['vote'].map(vote_recode)

# Merge issue flags from resolutions
ISSUE_COLS = list(ISSUES.values())
df = df.merge(res_raw[['vote_id'] + ISSUE_COLS], on='vote_id', how='left')
print('After merging issue flags:', df.shape)
df.head()

In [ ]:
# Rename for clarity
df = df.rename(columns={
    'vote_id': 'rcid',
    'state_code': 'ccode',
    'state_name': 'country',
    'assembly_session': 'session',
    'significant_vote': 'importantvote',
    'colonization': 'co',
    'human_rights': 'hr',
    'israel_palestine': 'me',
    'disarmament': 'di',
    'nuclear_weapons': 'nu',
    'economic_development': 'ec'
})

# Add short issue codes (fill NaN with 0)
for col in ['co','hr','me','di','nu','ec']:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

print('Final columns:', df.columns.tolist())
print('Countries:', df['country'].nunique())
print('Resolutions:', df['rcid'].nunique())
print('Year range:', df['year'].min(), '–', df['year'].max())
df.head(3)

In [ ]:
# Save master cleaned file
MASTER = os.path.join(PROC, 'votes_clean.csv')
df.to_csv(MASTER, index=False)
print('Saved:', MASTER)

## Step 1.4 – Temporal Slices

In [ ]:
ERAS = {
    'full':        (1946, 2015),
    'cold_war':    (1946, 1991),
    'post_cw':     (1991, 2001),
    'post_9_11':   (2001, 2014),
    'recent':      (2014, 2015),   # Kaggle only goes to 2015
}

era_dfs = {}
for name, (y_start, y_end) in ERAS.items():
    era_dfs[name] = df[(df['year'] >= y_start) & (df['year'] <= y_end)].copy()
    out = os.path.join(PROC, f'votes_{name}.csv')
    era_dfs[name].to_csv(out, index=False)
    print(f'{name:15s}: {era_dfs[name].shape[0]:>9,} rows | '
          f'{era_dfs[name]["country"].nunique():>4} countries | '
          f'{era_dfs[name]["rcid"].nunique():>5} resolutions')

## Step 1.5 – External Attributes: UN Regional Groups

In [ ]:
# UN Regional Group mapping (hardcoded from un.org)
un_regional_groups = {
    # African Group
    'Algeria':'Africa','Angola':'Africa','Benin':'Africa','Botswana':'Africa',
    'Burkina Faso':'Africa','Burundi':'Africa','Cabo Verde':'Africa','Cameroon':'Africa',
    'Central African Republic':'Africa','Chad':'Africa','Comoros':'Africa',
    "Côte d'Ivoire":'Africa','Djibouti':'Africa','Egypt':'Africa',
    'Equatorial Guinea':'Africa','Eritrea':'Africa','Eswatini':'Africa',
    'Ethiopia':'Africa','Gabon':'Africa','Gambia':'Africa','Ghana':'Africa',
    'Guinea':'Africa','Guinea-Bissau':'Africa','Kenya':'Africa','Lesotho':'Africa',
    'Liberia':'Africa','Libya':'Africa','Madagascar':'Africa','Malawi':'Africa',
    'Mali':'Africa','Mauritania':'Africa','Mauritius':'Africa','Morocco':'Africa',
    'Mozambique':'Africa','Namibia':'Africa','Niger':'Africa','Nigeria':'Africa',
    'Rwanda':'Africa','Sao Tome and Principe':'Africa','Senegal':'Africa',
    'Seychelles':'Africa','Sierra Leone':'Africa','Somalia':'Africa',
    'South Africa':'Africa','South Sudan':'Africa','Sudan':'Africa',
    'Togo':'Africa','Tunisia':'Africa','Uganda':'Africa',
    'United Republic of Tanzania':'Africa','Zambia':'Africa','Zimbabwe':'Africa',
    # Asia-Pacific
    'Afghanistan':'AsiaPacific','Bangladesh':'AsiaPacific','Bhutan':'AsiaPacific',
    'Cambodia':'AsiaPacific','China':'AsiaPacific','Fiji':'AsiaPacific',
    'India':'AsiaPacific','Indonesia':'AsiaPacific','Japan':'AsiaPacific',
    'Kazakhstan':'AsiaPacific','Kyrgyzstan':'AsiaPacific','Lao People Democratic Republic':'AsiaPacific',
    'Malaysia':'AsiaPacific','Maldives':'AsiaPacific','Marshall Islands':'AsiaPacific',
    'Micronesia':'AsiaPacific','Mongolia':'AsiaPacific','Myanmar':'AsiaPacific',
    'Nauru':'AsiaPacific','Nepal':'AsiaPacific','New Zealand':'AsiaPacific',
    'North Korea':'AsiaPacific','Pakistan':'AsiaPacific','Palau':'AsiaPacific',
    'Papua New Guinea':'AsiaPacific','Philippines':'AsiaPacific','Republic of Korea':'AsiaPacific',
    'Samoa':'AsiaPacific','Singapore':'AsiaPacific','Solomon Islands':'AsiaPacific',
    'Sri Lanka':'AsiaPacific','Tajikistan':'AsiaPacific','Thailand':'AsiaPacific',
    'Timor-Leste':'AsiaPacific','Tonga':'AsiaPacific','Turkmenistan':'AsiaPacific',
    'Tuvalu':'AsiaPacific','Uzbekistan':'AsiaPacific','Vanuatu':'AsiaPacific',
    'Viet Nam':'AsiaPacific','Australia':'AsiaPacific',
    # Eastern European
    'Albania':'EasternEurope','Armenia':'EasternEurope','Azerbaijan':'EasternEurope',
    'Belarus':'EasternEurope','Bosnia and Herzegovina':'EasternEurope',
    'Bulgaria':'EasternEurope','Croatia':'EasternEurope','Czechia':'EasternEurope',
    'Estonia':'EasternEurope','Georgia':'EasternEurope','Hungary':'EasternEurope',
    'Latvia':'EasternEurope','Lithuania':'EasternEurope','Moldova':'EasternEurope',
    'Montenegro':'EasternEurope','North Macedonia':'EasternEurope','Poland':'EasternEurope',
    'Romania':'EasternEurope','Russia':'EasternEurope','Serbia':'EasternEurope',
    'Slovakia':'EasternEurope','Slovenia':'EasternEurope','Ukraine':'EasternEurope',
    # GRULAC (Latin America & Caribbean)
    'Antigua and Barbuda':'GRULAC','Argentina':'GRULAC','Bahamas':'GRULAC',
    'Barbados':'GRULAC','Belize':'GRULAC','Bolivia':'GRULAC','Brazil':'GRULAC',
    'Chile':'GRULAC','Colombia':'GRULAC','Costa Rica':'GRULAC','Cuba':'GRULAC',
    'Dominica':'GRULAC','Dominican Republic':'GRULAC','Ecuador':'GRULAC',
    'El Salvador':'GRULAC','Grenada':'GRULAC','Guatemala':'GRULAC','Guyana':'GRULAC',
    'Haiti':'GRULAC','Honduras':'GRULAC','Jamaica':'GRULAC','Mexico':'GRULAC',
    'Nicaragua':'GRULAC','Panama':'GRULAC','Paraguay':'GRULAC','Peru':'GRULAC',
    'Saint Kitts and Nevis':'GRULAC','Saint Lucia':'GRULAC',
    'Saint Vincent and the Grenadines':'GRULAC','Suriname':'GRULAC',
    'Trinidad and Tobago':'GRULAC','Uruguay':'GRULAC','Venezuela':'GRULAC',
    # WEOG (Western Europe & Others)
    'Andorra':'WEOG','Austria':'WEOG','Belgium':'WEOG','Canada':'WEOG',
    'Cyprus':'WEOG','Denmark':'WEOG','Finland':'WEOG','France':'WEOG',
    'Germany':'WEOG','Greece':'WEOG','Iceland':'WEOG','Ireland':'WEOG',
    'Israel':'WEOG','Italy':'WEOG','Liechtenstein':'WEOG','Luxembourg':'WEOG',
    'Malta':'WEOG','Monaco':'WEOG','Netherlands':'WEOG','Norway':'WEOG',
    'Portugal':'WEOG','San Marino':'WEOG','Spain':'WEOG','Sweden':'WEOG',
    'Switzerland':'WEOG','Turkey':'WEOG','United Kingdom':'WEOG',
    'United States of America':'WEOG',
    # Arab States (sub-group often within Africa/Asia but distinct voting bloc)
    'Bahrain':'ArabStates','Iraq':'ArabStates','Jordan':'ArabStates',
    'Kuwait':'ArabStates','Lebanon':'ArabStates','Oman':'ArabStates',
    'Qatar':'ArabStates','Saudi Arabia':'ArabStates','Syria':'ArabStates',
    'United Arab Emirates':'ArabStates','Yemen':'ArabStates',
    "Iran (Islamic Republic of)":'AsiaPacific','Iran':'AsiaPacific',
}

region_df = pd.DataFrame(list(un_regional_groups.items()), columns=['country','region'])
out = os.path.join(EXT, 'un_regional_groups.csv')
region_df.to_csv(out, index=False)
print(f'Saved {len(region_df)} country-region mappings to {out}')
region_df['region'].value_counts()

In [ ]:
# World Bank income classification (approximation from WB 2023 data)
income_groups = {
    'High income': [
        'United States of America','Canada','United Kingdom','France','Germany',
        'Japan','Australia','Italy','Spain','Netherlands','Sweden','Norway',
        'Denmark','Finland','Austria','Belgium','Switzerland','New Zealand',
        'Republic of Korea','Israel','Singapore','United Arab Emirates',
        'Qatar','Kuwait','Bahrain','Saudi Arabia','Oman','Iceland','Ireland',
        'Luxembourg','Portugal','Greece','Cyprus','Malta','Czechia','Estonia',
        'Latvia','Lithuania','Slovakia','Slovenia','Hungary','Poland',
        'Croatia','Romania','Bulgaria','Andorra','Monaco','Liechtenstein',
        'San Marino','Brunei Darussalam','Bahamas','Barbados','Trinidad and Tobago',
        'Chile','Panama','Uruguay'
    ],
    'Upper middle income': [
        'China','Brazil','Russia','Mexico','Argentina','Colombia','Peru',
        'South Africa','Turkey','Thailand','Malaysia','Iran','Iraq','Jordan',
        'Cuba','Dominican Republic','Ecuador','Guatemala','Venezuela',
        'Algeria','Libya','Gabon','Botswana','Namibia','Mauritius',
        'Belarus','Serbia','Montenegro','North Macedonia','Albania','Bosnia and Herzegovina',
        'Azerbaijan','Armenia','Georgia','Kazakhstan','Turkmenistan',
        'Jamaica','Paraguay','Suriname','Fiji','Tonga','Samoa'
    ],
    'Lower middle income': [
        'India','Pakistan','Bangladesh','Sri Lanka','Philippines','Indonesia',
        'Viet Nam','Myanmar','Cambodia','Lao People Democratic Republic',
        'Nepal','Bhutan','Mongolia','Timor-Leste','Papua New Guinea',
        'Egypt','Morocco','Tunisia','Sudan','Nigeria','Ghana',
        'Kenya','Tanzania','Zambia','Zimbabwe','Uganda','Ethiopia',
        'Senegal','Cameroon','Ivory Coast','Mali','Mauritania',
        'Honduras','Nicaragua','El Salvador','Bolivia','Guyana',
        'Uzbekistan','Kyrgyzstan','Tajikistan','Moldova','Ukraine',
        'Lebanon','Syria','Yemen'
    ],
    'Low income': [
        'Afghanistan','Haiti','Somalia','South Sudan','Chad','Niger',
        'Burkina Faso','Guinea','Guinea-Bissau','Liberia','Sierra Leone',
        'Central African Republic','Eritrea','Rwanda','Burundi','Malawi',
        'Mozambique','Togo','Benin','Madagascar','Democratic Republic of the Congo',
        'Congo','Angola','Djibouti','Comoros','Lesotho','Eswatini',
        'Gambia','Sao Tome and Principe','Equatorial Guinea',
        'North Korea','Kyrgyzstan'
    ]
}

rows = []
for income, countries in income_groups.items():
    for c in countries:
        rows.append({'country': c, 'income_group': income})

income_df = pd.DataFrame(rows).drop_duplicates('country')
out2 = os.path.join(EXT, 'wb_income_groups.csv')
income_df.to_csv(out2, index=False)
print(f'Saved {len(income_df)} country income classifications to {out2}')
income_df['income_group'].value_counts()

In [ ]:
# Basic EDA plot — vote distribution over time
yearly = df.groupby('year')['v'].value_counts(normalize=True).unstack(fill_value=0)
yearly.columns = ['Abstain (0)', 'No (-1)', 'Yes (1)'] if -1 in yearly.columns else yearly.columns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22')

# Resolutions per year
rcid_per_year = df.groupby('year')['rcid'].nunique()
axes[0].bar(rcid_per_year.index, rcid_per_year.values, color='#58a6ff', alpha=0.8)
axes[0].set_xlabel('Year', color='white')
axes[0].set_ylabel('Resolutions', color='white')
axes[0].set_title('Resolutions per Year', color='white')
axes[0].tick_params(colors='white')
for spine in axes[0].spines.values(): spine.set_edgecolor('#30363d')

# Vote distribution over time
yearly_v = df.groupby('year')['v'].mean()
axes[1].plot(yearly_v.index, yearly_v.values, color='#3fb950', lw=2)
axes[1].axhline(0, color='#f85149', lw=0.8, ls='--')
axes[1].set_xlabel('Year', color='white')
axes[1].set_ylabel('Mean vote (1=yes, 0=abstain, -1=no)', color='white')
axes[1].set_title('Mean Vote Value Over Time', color='white')
axes[1].tick_params(colors='white')
for spine in axes[1].spines.values(): spine.set_edgecolor('#30363d')

plt.tight_layout()
out_fig = os.path.join(PLOTS, 'p1_vote_overview.png')
plt.savefig(out_fig, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved:', out_fig)

In [ ]:
# Summary table
summary = []
for name, (y0, y1) in ERAS.items():
    sub = era_dfs[name]
    summary.append({
        'era': name, 'years': f'{y0}–{y1}',
        'n_rows': len(sub), 'n_countries': sub['country'].nunique(),
        'n_resolutions': sub['rcid'].nunique()
    })
summary_df = pd.DataFrame(summary)
out3 = os.path.join(TABLES, 'p1_era_summary.csv')
summary_df.to_csv(out3, index=False)
print(summary_df.to_string(index=False))

## ✅ Phase 1 Complete
**Outputs:**
- `data/processed/votes_clean.csv` — master cleaned vote table
- `data/processed/votes_{era}.csv` — 5 temporal slices
- `data/external/un_regional_groups.csv` — country → UN region
- `data/external/wb_income_groups.csv` — country → income group
- `results/plots/p1_vote_overview.png`
- `results/tables/p1_era_summary.csv`

**→ Proceed to Notebook 02: Network Construction**